In [0]:
dbutils.secrets.listScopes()

In [0]:
params = {
    "series_id": "UNRATE",
    "api_key": FRED_API_KEY,
    "file_type": "json"
}

In [0]:
# 1. Imports

import requests
from pyspark.sql import functions as F

In [0]:
# 2. Configuration

SERIES_ID = "UNRATE"

CATALOG = "workspace"
SCHEMA = "macro"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_unrate"

FRED_API_URL = "https://api.stlouisfed.org/fred/series/observations"

In [0]:
# 4. Extract data from FRED

params = {
    "series_id": SERIES_ID,
    "api_key": api_key,
    "file_type": "json",
}

response = requests.get(
    FRED_API_URL,
    params=params,
    timeout=30
)

response.raise_for_status()

data = response.json()

In [0]:
# 5. Convert FRED response to Spark DataFrame

observations = data["observations"]

raw_df = spark.createDataFrame(observations)

display(raw_df)

In [0]:
# 6. Select required columns

bronze_df = (
    raw_df
    .select(
        F.lit(SERIES_ID).alias("series_id"),
        F.to_date("date").alias("date"),
        F.expr("try_cast(value AS DOUBLE)").alias("value"),
    )
)

display(bronze_df)

In [0]:
# 7. Basic validation

print(f"Number of rows: {bronze_df.count()}")

print("Date range:")
bronze_df.select(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date")
).show()

print("Null values:")
bronze_df.select(
    F.sum(F.col("date").isNull().cast("int")).alias("null_dates"),
    F.sum(F.col("value").isNull().cast("int")).alias("null_values")
).show()

In [0]:
# 8. Create schema if necessary

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}
""")


In [0]:
# 9. Write Bronze Delta Table

(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)


In [0]:
# 10. Verify Bronze Table

display(
    spark.table(BRONZE_TABLE)
    .orderBy("date")
)

In [0]:
spark.sql("SHOW TABLES IN workspace.macro").show()

In [0]:
bronze = spark.table("workspace.macro.bronze_unrate")

print("Rows:", bronze.count())

display(
    bronze.orderBy("date")
)

In [0]:
import matplotlib.pyplot as plt

plot_df = (
    bronze_df
    .filter(F.col("value").isNotNull())
    .orderBy("date")
    .toPandas()
)

plt.figure(figsize=(14, 5))
plt.plot(plot_df["date"], plot_df["value"])
plt.title("U.S. Unemployment Rate (UNRATE)")
plt.xlabel("Date")
plt.ylabel("Unemployment Rate (%)")
plt.grid(True)
plt.show()